# Sprint 2 (v2): fix the flood label, then retrain

Sprint 2 v1 converged to **train IoU 0.77 but validation at the flood-everywhere baseline (~0.31)** — textbook overfitting, but the post-mortem showed the deeper cause is the **label**, not the network:

- A pixel classifier reached only AUC ~0.65–0.71 between the two locations, and the U-Net scored *lower* (0.57).
- Permutation importance put the signal in the **SWIR/NIR optical bands** (land cover), with terrain contributing almost nothing.
- Flooded fractions of 19% and 32% are implausibly high for inundation.

The v1 label was a **dry-vs-rainy-season Otsu threshold** on a single VV minimum composite. That cannot separate *new water* from *a surface that is simply darker in the rainy season*, so the U-Net learned land cover.

**What changes in v2**

1. **The label is rebuilt as bi-temporal change detection around a dated flood event** (the April–May 2024 Nairobi floods). Open water is specular at C-band, so new inundation is a sharp VV backscatter *drop* between a pre-event and an event-peak composite. Anything stable across the two dates (land cover) cancels out. Physical refinements (slope, permanent water, speckle) follow UN-SPIDER practice. See `src/utils/flood_label.py` / `src/utils/gee_utils.get_event_flood_label`.
2. **The label is validated against an independent reference** (NRRP high-risk zones, or an OSM river-corridor buffer as fallback) with a *lift* statistic, so "is this actually flooding?" gets a number, not a hand-wave. See `src/utils/label_validation.py`.
3. **The U-Net can optionally see the Sentinel-1 modality its label comes from.** v1's 9 inputs were optical + terrain only — no SAR — so it was structurally forced to infer water from land cover. The `EVENT_S1_LAYOUT` adds the pre/event VV bands (11 inputs), the Sen1Floods11 "VV in, water out" paradigm extended bi-temporally.
4. **Regularisation against the single-location overfitting**: dropout, weight decay, and early stopping.

Everything else (leakage-aware spatial holdout, overlap-masked evaluation) carries over from v1.

## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** (do this first; it restarts the session).
- The Sprint 2 v2 source (`src/utils/flood_label.py`, `label_validation.py`, `src/models/diagnostics.py`, updated `gee_utils.py`/`dataset.py`) must be pushed to GitHub. The repo-check cell verifies this.
- You need an Earth Engine–enabled GCP project to re-export the raster stack with the new event label.
- Set the config knobs in the next-but-one cell (`GEE_PROJECT_ID`, `INCLUDE_S1_INPUTS`, event dates live in `src/config.py`).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. This dataset is small enough to train on CPU, but a T4 is much faster.")

In [ ]:
%%bash
cd /content
if [ -d nairobi-risk-mapping ]; then
  cd nairobi-risk-mapping && git pull origin main
else
  git clone https://github.com/emmanuelamor5/nairobi-risk-mapping.git
fi

In [ ]:
%cd /content/nairobi-risk-mapping

from pathlib import Path
required = [
    "src/config.py", "src/data_pipeline.py",
    "src/utils/gee_utils.py", "src/utils/flood_label.py", "src/utils/label_validation.py",
    "src/models/attention_unet.py", "src/models/attention_unet_model.py", "src/models/dataset.py",
    "src/models/losses.py", "src/models/metrics.py", "src/models/transfer_learning.py",
    "src/models/evaluation.py", "src/models/diagnostics.py",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, (
    f"Not in the cloned repo: {missing}. Commit and push the Sprint 2 v2 files from your machine, "
    "then rerun from the top."
)
print("Repository check passed.")

In [ ]:
!pip install -q -r requirements.txt

## 1. Configuration

One place for every knob. The event dates and change-detection thresholds live in `src/config.py` (`FLOOD_EVENT_*`, `FLOOD_CHANGE_THRESHOLD_DB`, `FLOOD_MAX_SLOPE_DEG`, ...); change them there if you anchor the label to a different documented flood.

In [ ]:
# --- Earth Engine ---
GEE_PROJECT_ID = ""          # <-- your EE-enabled GCP project id

# --- Label / layout ---
LABEL_MODE = "event"          # "event" (v2 change-detection) or "seasonal" (v1, for comparison)
INCLUDE_S1_INPUTS = True      # True -> 11-band EVENT_S1 layout (U-Net sees pre/event VV); False -> 9-band v1 layout
ALSO_EXPORT_SEASONAL = True   # also export the old seasonal-label stack so the two labels can be compared

# --- Label validation reference ---
NRRP_GEOJSON_PATH = None      # path to NRRP high-risk zone polygons (GeoJSON/shapefile) in Drive; None -> OSM river-corridor fallback
RIVER_BUFFER_M = 120          # buffer around OSM rivers/streams when NRRP polygons are unavailable

# --- Model / training ---
PRETRAINED_CHECKPOINT_PATH = None   # a .pt saved from THIS AttentionUNet architecture, or None to train from scratch
DROPOUT = 0.2
WEIGHT_DECAY = 1e-4
EPOCHS = 60
EARLY_STOPPING_PATIENCE = 12
BASE_CHANNELS = 32
SEED = 42

import logging, random
import numpy as np, torch
logging.basicConfig(level=logging.INFO, force=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from src.models.dataset import DEFAULT_LAYOUT, EVENT_S1_LAYOUT
layout = EVENT_S1_LAYOUT if INCLUDE_S1_INPUTS else DEFAULT_LAYOUT
band_names = list(layout.band_names)
print(f"Layout: {'EVENT_S1' if INCLUDE_S1_INPUTS else 'DEFAULT'} | {layout.in_channels} input channels | "
      f"label band index {layout.label_index}")
print("Bands:", band_names)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/nairobi_risk_mapping"
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

## 2. Rebuild the flood label (Earth Engine export)

This kicks off an **asynchronous** `Export.image.toDrive` for the event-label stack (and, if `ALSO_EXPORT_SEASONAL`, the old seasonal stack for comparison). Exports run on Google's servers and continue even if Colab disconnects.

If you already have a fresh event export in Drive, you can skip this cell and go to section 3 — it locates the newest matching file.

In [ ]:
import ee
from src.data_pipeline import DataPipeline
from src.utils import gee_utils

pipeline = DataPipeline(settlement_key="kibera", gee_project_id=GEE_PROJECT_ID or None)

tasks = {}
event_task = pipeline.acquire_raster_stack(label_mode=LABEL_MODE, include_s1_inputs=INCLUDE_S1_INPUTS)
tasks[LABEL_MODE] = event_task

event_variant = LABEL_MODE + ("_s1" if INCLUDE_S1_INPUTS else "")
print(f"Started export '{LABEL_MODE}' label -> kibera_raster_stack_{event_variant}.tif "
      f"({len(band_names)} bands, include_s1_inputs={INCLUDE_S1_INPUTS}).")

if ALSO_EXPORT_SEASONAL and LABEL_MODE == "event":
    seasonal_task = pipeline.acquire_raster_stack(label_mode="seasonal", include_s1_inputs=False)
    tasks["seasonal"] = seasonal_task
    print("Started export 'seasonal' label -> kibera_raster_stack_seasonal.tif (10 bands, comparison).")

print("\nEarth Engine exports run on Google's servers and can take several to tens of minutes.")
print("Run the NEXT cell and let it block until the tasks report COMPLETED before going to section 3.")

In [ ]:
import time
# Blocks until every export reaches a terminal state, printing status as it goes.
# Safe to re-run: if the tasks are already done it returns almost immediately.
TERMINAL = {"COMPLETED", "FAILED", "CANCELLED", "CANCEL_REQUESTED"}
while True:
    states = {name: task.status().get("state") for name, task in tasks.items()}
    print(time.strftime("%H:%M:%S"), states, flush=True)
    if all(s in TERMINAL for s in states.values()):
        break
    time.sleep(30)

for name, task in tasks.items():
    st = task.status()
    line = f"{name:10s} {st.get('state')}"
    if st.get("state") == "FAILED":
        line += f"  ERROR: {st.get('error_message', '(no message)')}"
    print(line)

failed = [n for n, t in tasks.items() if t.status().get("state") == "FAILED"]
assert not failed, (
    f"Export(s) failed: {failed}. Read the ERROR line above. A common cause is no Sentinel-1 "
    f"coverage in the event window over the geometry -- widen FLOOD_EVENT_START/END in config.py."
)
print("\nAll exports completed. Give Drive a few seconds to sync, then run section 3.")

## 3. Locate and verify the exported stack

Earth Engine never overwrites an export, so repeated runs leave `(1).tif`, `(2).tif`, ... This finds the newest file matching the chosen label mode and checks its band count against the selected layout.

In [ ]:
import glob, os
expected_bands = len(band_names)

candidates = sorted(glob.glob(f"{DRIVE_DIR}/kibera_raster_stack*.tif"), key=os.path.getmtime, reverse=True)
assert candidates, (
    f"No kibera_raster_stack*.tif in {DRIVE_DIR} yet. Run section 2 and wait for the export "
    f"(the poll cell) to report COMPLETED, then re-run this cell."
)

import rasterio
print(f"Looking for a {expected_bands}-band stack (the {'EVENT_S1' if INCLUDE_S1_INPUTS else 'DEFAULT'} layout).\n")
band_counts = {}
stack_path = None
for p in candidates:
    with rasterio.open(p) as src:
        nbands = src.count
    band_counts[p] = nbands
    flag = "  <-- using this" if (stack_path is None and nbands == expected_bands) else ""
    print(f"{os.path.basename(p):44s} {nbands} bands  {time.ctime(os.path.getmtime(p))}{flag}")
    if stack_path is None and nbands == expected_bands:
        stack_path = p

if stack_path is None:
    only_10 = all(v == 10 for v in band_counts.values())
    if INCLUDE_S1_INPUTS and only_10:
        hint = ("The file(s) above are 10-band stacks -- your OLD v1 seasonal export (the wrong label "
                "for v2). The 12-band event export has not appeared yet: run section 2, wait for the "
                "poll cell to say COMPLETED, let Drive sync, then re-run this cell. (Or, to train on "
                "the event label WITHOUT the Sentinel-1 input bands, set INCLUDE_S1_INPUTS=False in "
                "section 1 and export a 10-band event stack -- note your current 10-band file is the "
                "seasonal label, not the event label.)")
    else:
        hint = (f"Run section 2 with INCLUDE_S1_INPUTS={INCLUDE_S1_INPUTS} and wait for the export to finish.")
    raise AssertionError(f"No export with {expected_bands} bands found. {hint}")

print("\nUsing:", stack_path)

In [ ]:
label_idx = layout.label_index
with rasterio.open(stack_path) as src:
    data = src.read()
    stack_transform, stack_crs = src.transform, src.crs
    print("Dimensions (h x w):", src.height, "x", src.width, "| bands:", src.count, "| CRS:", src.crs)

assert data.shape[0] == expected_bands, f"Expected {expected_bands} bands, found {data.shape[0]}"
assert min(data.shape[1:]) >= 256, "Raster smaller than one 256px tile; widen the export buffer."

print("\nPer-band NaN fraction:")
for name, band in zip(band_names, data):
    print(f"  {name:12s} {np.isnan(band).mean():.4f}")

label = data[label_idx]
finite = label[np.isfinite(label)]
frac = float(finite.mean())
print(f"\nflood_label unique values: {np.unique(finite).tolist()}")
print(f"flood_label flooded fraction over the whole raster: {frac:.3f}")
assert set(np.unique(finite).tolist()) <= {0.0, 1.0}, "flood_label is not binary"
assert 0.001 < frac < 0.99, "flood_label is near-constant; the model cannot learn from it"
if frac > 0.15:
    print("NOTE: flooded fraction is still high (>15%). Compare against the reference in section 5 "
          "and consider tightening FLOOD_CHANGE_THRESHOLD_DB or the event window in config.py.")

## 4. Inspect the new label (and compare to the old one)

Side-by-side RGB / event label / seasonal label. The event label should be sparser and hug the river corridors; the seasonal label tended to paint vegetation and dry ground.

In [ ]:
import matplotlib.pyplot as plt

def find_stack_with_bands(n):
    for p in candidates:
        with rasterio.open(p) as s:
            if s.count == n:
                return p
    return None

rgb = np.nan_to_num(np.stack([data[2], data[1], data[0]], axis=-1))
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-9)

# locate a seasonal stack (10 bands, seasonal export) for comparison, if present
seasonal_path = None
for p in candidates:
    if p == stack_path:
        continue
    with rasterio.open(p) as s:
        if s.count == 10:
            seasonal_path = p
            break

ncols = 3 if seasonal_path else 2
fig, ax = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
ax[0].imshow(rgb); ax[0].set_title("RGB (S2)")
ax[1].imshow(np.nan_to_num(label), cmap="Blues", vmin=0, vmax=1)
ax[1].set_title(f"event label ({frac*100:.1f}% flooded)")
if seasonal_path:
    with rasterio.open(seasonal_path) as s:
        seas = s.read(7)  # seasonal 10-band layout: flood_label at band 7 (1-indexed)
    ax[2].imshow(np.nan_to_num(seas), cmap="Oranges", vmin=0, vmax=1)
    ax[2].set_title(f"seasonal label ({np.nanmean(seas)*100:.1f}% flooded)")
plt.tight_layout(); plt.show()

## 5. Validate the label against an independent reference

The question v1 left open: *is this flooding, or land cover?* We check how strongly the labelled pixels concentrate inside an independent high-risk reference, via **lift** = (flooded pixels inside the reference) / (all valid pixels inside the reference). Lift ≫ 1 is what a real flood label should show; lift ≈ 1 is the land-cover-proxy signature.

**Reference priority:** NRRP high-risk zone polygons if available, else an OSM river-corridor buffer.

### Getting the NRRP 37 high-risk zones into Drive

1. Obtain the zone polygons as a vector file. In order of preference:
   - The official **NRRP / NRRC** or **Nairobi City County (NCC) GIS** riparian high-risk zone layer (shapefile or GeoJSON), if you can request it.
   - A **digitised** version: trace the 37 zones from the published NRRP map into QGIS/geojson.io and export GeoJSON.
   - If neither is attainable, use the **statutory riparian reserve** as a defensible proxy: the OSM river-corridor fallback below buffered to the legal reserve width (set `RIVER_BUFFER_M = 30` or `60` in section 1 per the 30–60 m riparian buffer the proposal cites). State clearly in the write-up that this is a riparian-reserve proxy, not the gazetted NRRP polygons.
2. Upload the file to the Drive folder `MyDrive/nairobi_risk_mapping/` (e.g. `nrrp_high_risk_zones.geojson`, or a zipped shapefile `nrrp.zip`).
3. Either set `NRRP_GEOJSON_PATH` in section 1 to its full path, or just name it starting with `nrrp` — the next cell auto-discovers `nrrp*` files in the Drive folder. Supported: `.geojson/.json`, `.shp`, zipped shapefile `.zip`, `.gpkg`, `.kml`.

In [ ]:
from src.utils import label_validation
import geopandas as gpd
import glob

out_shape = data.shape[1:]
valid_mask = np.isfinite(label)

# Resolve the NRRP file: explicit path wins; otherwise auto-discover nrrp* in Drive.
nrrp_path = NRRP_GEOJSON_PATH
if not nrrp_path:
    found = []
    for ext in ("geojson", "json", "shp", "zip", "gpkg", "kml"):
        found += glob.glob(f"{DRIVE_DIR}/[Nn][Rr][Rr][Pp]*.{ext}")
    if found:
        nrrp_path = sorted(found)[0]
        print(f"Auto-discovered NRRP reference: {nrrp_path}")

reference = None
ref_source = None
if nrrp_path:
    try:
        gdf = label_validation.load_reference_polygons(nrrp_path, target_crs=stack_crs)
        reference = label_validation.rasterize_reference(gdf, stack_transform, out_shape, stack_crs)
        ref_source = f"NRRP polygons ({len(gdf)} zones, {nrrp_path})"
    except Exception as e:
        print("Failed to load NRRP polygons, falling back to OSM rivers:", e)

if reference is None:
    import osmnx as ox
    from shapely.geometry import box
    with rasterio.open(stack_path) as src:
        b = src.bounds
    aoi = gpd.GeoDataFrame(geometry=[box(b.left, b.bottom, b.right, b.top)], crs=stack_crs).to_crs(4326)
    try:
        waterways = ox.features_from_polygon(aoi.geometry.iloc[0], tags={"waterway": ["river", "stream", "drain"]})
        waterways = waterways[waterways.geometry.type.isin(["LineString", "MultiLineString"])]
        reference = label_validation.river_corridor_reference(
            waterways.to_crs(stack_crs), stack_transform, out_shape, stack_crs, buffer_m=RIVER_BUFFER_M)
        ref_source = f"OSM river corridor (buffer {RIVER_BUFFER_M} m) -- riparian-reserve proxy, not gazetted NRRP zones"
    except Exception as e:
        print("Could not build an OSM river reference:", e)

if reference is not None:
    print(f"Reference: {ref_source} | covers {reference.mean()*100:.1f}% of the scene\n")
    labels_to_check = {"event_v2": np.nan_to_num(label)}
    if seasonal_path:
        with rasterio.open(seasonal_path) as s:
            labels_to_check["seasonal_v1"] = np.nan_to_num(s.read(7))
    results = label_validation.compare_labels(labels_to_check, reference, valid_mask)
    for name, m in results.items():
        print(f"{name:12s} lift={m['lift']:.2f}  precision={m['precision']:.2f}  "
              f"recall={m['recall']:.2f}  iou={m['iou']:.2f}  flooded_frac={m['label_fraction']:.3f}")
    print("\nInterpretation: a higher lift for event_v2 than seasonal_v1 is direct evidence the new label "
          "concentrates in genuinely flood-prone terrain rather than tracking land cover. Note that OSM "
          "waterway coverage in informal settlements is incomplete, so the river-corridor lift is a FLOOR "
          "-- the NRRP polygons give the authoritative number. If the event label's lift is still ~1, the "
          "label is NOT yet trustworthy -- revisit the event window / threshold before quoting any IoU as "
          "flood-prediction accuracy.")
else:
    print("No reference available -- supply NRRP_GEOJSON_PATH to validate the label quantitatively.")

## 6. Tile, split, and measure the overlap

Rebuild tiles from the exported raster (deterministic, seconds). SAR bands are normalised by a fixed dB window so the pre-vs-event drop survives (see `DataPipeline._normalise_bands`). Then hold out one whole location and measure the 50px shared strip, exactly as in v1.

In [ ]:
import shutil, json
from src.data_pipeline import DataPipeline
from src.models.dataset import spatial_holdout_split
from src.models.evaluation import validation_overlap_mask

shutil.rmtree("data/tiles/kibera", ignore_errors=True)
pipe = DataPipeline(settlement_key="kibera")
manifest_path = str(pipe.tile_raster_stack(stack_path, split="train", band_names=band_names))
manifest = json.load(open(manifest_path))
locs = sorted({(t["row_off"], t["col_off"]) for t in manifest})
print(f"{len(manifest)} tiles at {len(locs)} distinct locations: {locs}")
assert len(locs) >= 2, "Need >=2 locations to hold one out. Widen the export buffer."

split = spatial_holdout_split(manifest_path)
train_locs = sorted({(t["row_off"], t["col_off"]) for t in split["train"]})
val_loc = split["holdout_location"]
safe_mask = validation_overlap_mask(val_loc, train_locs, tile_size=256)
print(f"Train: {len(split['train'])} tiles at {train_locs}")
print(f"Val:   {len(split['val'])} tiles at {val_loc}")
print(f"Validation pixels usable leakage-free: {int(safe_mask.sum())} of {safe_mask.size} "
      f"({(~safe_mask).mean()*100:.1f}% shared with a training tile)")

In [ ]:
# Baseline to beat: predicting flood everywhere scores IoU = flooded fraction on the clean pixels.
val_orig = next(t for t in split["val"] if t["tile_id"].endswith("_orig"))
val_label = np.nan_to_num(np.load(val_orig["path"])[layout.label_index])
baseline_iou = float((val_label[safe_mask] >= 0.5).mean())
print(f"Flood-everywhere baseline IoU on leakage-free pixels: {baseline_iou:.3f}")

## 7. Model and training

`EVENT_S1_LAYOUT` makes the model's input channel count 11 automatically. Dropout, weight decay and early stopping are the v2 anti-overfitting additions. `PRETRAINED_CHECKPOINT_PATH=None` trains from scratch; the loader only accepts a checkpoint saved from this same `AttentionUNet`.

In [ ]:
from src.models.attention_unet_model import AttentionUNetModel
from src.models.transfer_learning import trainable_parameter_count

model = AttentionUNetModel(
    base_channels=BASE_CHANNELS,
    dropout=DROPOUT,
    pretrained_checkpoint=PRETRAINED_CHECKPOINT_PATH,
    freeze_pretrained_encoder=True,
    layout=layout,                 # sets in_channels = layout.in_channels
)
if model.transfer_report is not None:
    r = model.transfer_report
    print(f"Transferred {len(r['transferred'])} tensors; reinitialised: {r['reinitialised']}")
else:
    print("Training from scratch (no pretrained checkpoint).")
print(trainable_parameter_count(model.network))

In [ ]:
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints_v2"
result = model.fit(
    train_manifest_entries=split["train"],
    val_manifest_entries=split["val"],
    epochs=EPOCHS,
    batch_size=2,
    learning_rate=1e-3,
    checkpoint_dir=CHECKPOINT_DIR,
    weight_decay=WEIGHT_DECAY,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)
print(f"\nBest validation IoU during training (includes shared strip): {result['best_val_iou']:.4f}")
print(f"Stopped at epoch {result['stopped_epoch']} of {EPOCHS}.")

In [ ]:
history = result["history"]
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(ep, [h["train_loss"] for h in history], label="train")
ax[0].plot(ep, [h["val_loss"] for h in history], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(ep, [h["train_iou"] for h in history], label="train IoU")
ax[1].plot(ep, [h["val_iou"] for h in history], label="val IoU")
ax[1].axhline(0.60, color="red", ls="--", label="target 0.60")
ax[1].axhline(baseline_iou, color="grey", ls=":", label="flood-everywhere baseline")
ax[1].set_title("IoU"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 8. Leakage-free evaluation and prediction map

Score the best checkpoint only on validation pixels no training tile covers. The checkpoint was still selected on this location, so treat this as a convergence check, not a final metric.

In [ ]:
from src.models.evaluation import evaluate_masked

model.load_checkpoint(f"{CHECKPOINT_DIR}/attention_unet_best.pt")
clean = evaluate_masked(model, val_orig["path"], safe_mask, layout=layout)
print(f"During training (incl. shared strip): IoU {result['best_val_iou']:.3f}")
print(f"Leakage-free:                         IoU {clean['iou']:.3f}   F1 {clean['f1']:.3f}")
print(f"Flood-everywhere baseline:            IoU {clean['all_flooded_baseline_iou']:.3f}")
print(f"Pixels evaluated: {clean['evaluated_pixels']} of {clean['total_pixels']}")
beat = clean["iou"] > clean["all_flooded_baseline_iou"] + 0.02
print("\nBeats the trivial baseline:", "YES" if beat else "NO (still label- or capacity-limited)")
print("Sprint 2 convergence check (IoU >= 0.60):", "MET" if clean["iou"] >= 0.60 else "NOT MET")

In [ ]:
tile = np.load(val_orig["path"])
inputs = torch.from_numpy(np.nan_to_num(tile[layout.input_indices], nan=0.0)).float().unsqueeze(0)
proba = model.predict_proba(inputs)[0, 0].numpy()

fig, ax = plt.subplots(1, 4, figsize=(22, 6))
ax[0].imshow(np.nan_to_num(np.stack([tile[2], tile[1], tile[0]], axis=-1))); ax[0].set_title("RGB")
ax[1].imshow(np.nan_to_num(tile[layout.label_index]), cmap="Blues", vmin=0, vmax=1); ax[1].set_title("event label")
ax[2].imshow(proba, cmap="Blues", vmin=0, vmax=1); ax[2].set_title("predicted probability")
ax[3].imshow(safe_mask, cmap="gray"); ax[3].set_title("white = evaluated")
plt.show()

## 9. Did the signal move off land cover? (cross-location diagnostics)

Re-run the exact v1 probes on the new label: a cheap logistic pixel classifier trained on one location and scored on the other (cross-location AUC), plus permutation importance. Compare the numbers to v1 (AUC ~0.65–0.71, signal in SWIR/NIR). On the event label, the dependence should shift toward the Sentinel-1 / terrain bands and away from pure optical land cover.

In [ ]:
from src.models.diagnostics import cross_location_signal

train_orig = next(t for t in split["train"] if t["tile_id"].endswith("_orig"))
feature_names = [n for n in band_names if n != layout.label_name]
diag = cross_location_signal(
    np.load(train_orig["path"]), np.load(val_orig["path"]),
    layout.input_indices, layout.label_index, feature_names,
    subsample=40000, n_repeats=10, seed=SEED,
)
print(f"Cross-location pixel AUC: train {diag['train_auc']:.3f} | val {diag['val_auc']:.3f}")
print(f"Positive fraction: train {diag['train_positive_fraction']:.3f} | val {diag['val_positive_fraction']:.3f}")
print("\nPermutation importance (AUC drop when each band is shuffled):")
if diag["permutation_importance"]:
    for name, v in diag["permutation_importance"].items():
        print(f"  {name:12s} {v['mean']:+.4f} ± {v['std']:.4f}")

## 10. Save the run summary

In [ ]:
summary = {
    "label_mode": LABEL_MODE,
    "include_s1_inputs": INCLUDE_S1_INPUTS,
    "layout_in_channels": layout.in_channels,
    "flooded_fraction_raster": frac,
    "label_validation": results if reference is not None else None,
    "best_val_iou_during_training": result["best_val_iou"],
    "stopped_epoch": result["stopped_epoch"],
    "leakage_free": clean,
    "cross_location_diagnostics": diag,
    "trained_from_scratch": PRETRAINED_CHECKPOINT_PATH is None,
    "train_locations": train_locs,
    "val_location": list(val_loc),
    "raster_used": os.path.basename(stack_path),
    "dropout": DROPOUT, "weight_decay": WEIGHT_DECAY, "seed": SEED,
}
with open(f"{CHECKPOINT_DIR}/sprint2_v2_summary.json", "w") as f:
    json.dump({"summary": summary, "history": history}, f, indent=2, default=str)
print("Saved summary to", f"{CHECKPOINT_DIR}/sprint2_v2_summary.json")

## Sprint 2 v2 exit checklist

- [ ] Event label built from the dated flood window, flooded fraction plausible (compare to v1's 19–32%).
- [ ] **Label validated against NRRP / river corridor with lift > 1** — the label concentrates in flood-prone terrain, not land cover. This is the gate; a lift ≈ 1 means the label is still not trustworthy.
- [ ] Leakage-free IoU **clearly above the flood-everywhere baseline** (the v1 failure).
- [ ] Cross-location AUC and permutation importance recorded and compared to v1 (did the signal leave SWIR/NIR?).
- [ ] Results labelled trained-from-scratch vs transferred, and 9-band vs 11-band (S1-input) layout.
- [ ] `attention_unet_best.pt` and `sprint2_v2_summary.json` in Drive.

**If the label still fails validation (lift ≈ 1):** the event window or the change threshold is wrong, not the network. Re-anchor to a different dated flood (or tighten `FLOOD_CHANGE_THRESHOLD_DB`) before spending more on training. **If the label passes but IoU is still weak:** that is now a genuine modelling problem — try the 11-band S1 layout, a Sen1Floods11-compatible checkpoint, or raise capacity/augmentation.

Next: Sprint 3 — the GraphSAGE structural-deprivation head on the pruned Kibera graph. Also discuss with your supervisor whether the 0.60 IoU target and the cross-validation design should be amended given these results.